# Testing — shots: qlora

QLoRA adapters, seeds 42, 43, 44, with 1 and 3 shots. 1 shot = the first of the 3 few-shot examples; 0 shots are the existing runs of the plain arms, vanilla with 3 shots is the existing few-shot arm. GPU: 24 GB, disk 60 GB.

Only this part is enabled in the embedded `config.yml` below. Set `hf_token` / `vast_api_key` there (or as
environment variables in the template) and `pilot_problems: 20` for a pilot, then **Run All**. Results go to the
private dataset repo `output_repo`; runs already on the Hub are skipped; the machine stops itself at the end.

## 0. Configuration (edit here)

In [ ]:
%%writefile config.yml
# Which parts of the evaluation to run. Upload this file next to index.ipynb, edit, then Run All.
# Runs whose summary.json is already on the Hub are skipped, so rerunning resumes where it stopped.

arms:
  vanilla: false        # Qwen2.5-Coder-7B-Instruct, no fine-tuning
  few-shot: false       # vanilla + 3 examples in the prompt
  lora: false           # 3 adapters (seed 42, 43, 44)
  qlora: false          # 3 adapters on a 4-bit base; enable after QLoRA training finished
  qwen-14b: false       # Qwen2.5-Coder-14B-Instruct, needs a GPU with >= 40 GB
  # LoRA hyperparameter sensitivity, seed 42; enable after training/hyperparameter/index.ipynb pushed the adapters
  hp-lora-cosine: false     # cosine scheduler + 3% warmup
  hp-lora-r32: false        # rank 32, alpha 64
  hp-lora-lr1e-4: false     # learning rate 1e-4
  # Prompting shots: the same models with 1 or 3 training examples in the prompt (1 shot = first of the 3 examples).
  # 0 shots = the arms above; vanilla with 3 shots = few-shot.
  vanilla-1shot: false
  qwen-14b-1shot: false     # needs a GPU with >= 40 GB
  qwen-14b-3shot: false     # needs a GPU with >= 40 GB
  lora-1shot: false         # seeds 42, 43, 44
  lora-3shot: false
  qlora-1shot: true        # seeds 42, 43, 44
  qlora-3shot: true
  hp-lora-cosine-1shot: false   # seed 42
  hp-lora-cosine-3shot: false
  hp-lora-r32-1shot: false
  hp-lora-r32-3shot: false
  hp-lora-lr1e-4-1shot: false
  hp-lora-lr1e-4-3shot: false

refinement-loop:
  lora: false           # needs the lora results on the Hub
  qlora: false          # needs the qlora results on the Hub
  seed: auto           # auto = median seed by DS-1000 pass@1, or 42 / 43 / 44
  problems: 200        # proposal: 100-200 DS-1000 problems
  max_iterations: 3    # proposal: at most 3

retest:                # re-extract code from the replies on the Hub + rerun DS-1000 tests (no generation, CPU only)
  vanilla: false       # for runs made before the python_block fix; newer runs are skipped
  few-shot: false
  lora: false
  qlora: false
  qwen-14b: false

pilot_problems: null   # e.g. 20 for a quick pilot run; null = full evaluation

generation:            # identical for every arm
  ds1000_samples: 10   # pass@1 and pass@10 from 10 samples per problem
  ds1000_temperature: 0.8
  ds1000_top_p: 0.95
  max_new_tokens: 1024
  max_model_len: 8192
  seed: 0

# Credentials. Used only when the matching environment variable (HF_TOKEN / VAST_API_KEY) is not set.
# Keep these empty in any copy you share, commit or upload anywhere except the GPU machine.
hf_token: ""          # Hugging Face token with Write access (huggingface.co/settings/tokens)
vast_api_key: ""      # Vast.ai API key (cloud.vast.ai -> Keys -> API Keys); lets the machine stop itself

output_repo: dimsmauls/code4ml-eval-outputs   # private dataset repo for every result
dataset: dimsmauls/code4ml-annotated-rated
auto_shutdown: true    # stop the machine at the end (success or failure); needs VAST_API_KEY

## 1. Install
Generation: vLLM in the main environment. DS-1000 tests: the exact versions of the official `environment.yml`
(Python 3.10, pandas 1.5.3, numpy 1.26.4, torch 2.2.0 CPU, TensorFlow 2.16.1, ...) in a separate `ds1000-env/`
built with `uv`, because they conflict with vLLM. The official `execution.py` comes from DS-1000 at a pinned commit.

In [ ]:
%pip install -q -U vllm datasets huggingface_hub bitsandbytes pyyaml
%pip uninstall -y -q torchaudio
%pip install -q uv
!uv venv ds1000-env --python 3.10 -q --allow-existing
!uv pip install -q --python ds1000-env/bin/python numpy==1.26.4 pandas==1.5.3 scipy==1.12.0 scikit-learn==1.4.0 matplotlib==3.8.4 seaborn==0.13.2 statsmodels==0.14.1 xgboost==2.0.3 gensim==4.3.2 pyyaml tensorflow-cpu==2.16.1
!uv pip install -q --python ds1000-env/bin/python torch==2.2.0 --index-url https://download.pytorch.org/whl/cpu
!curl -sfL https://raw.githubusercontent.com/xlang-ai/DS-1000/b39aab71da6d23ef8d3cac59a7c5f834516ab334/execution.py -o execution.py && echo "execution.py ok"

## 2. Helper files
`evalkit.py` (prompts, official DS-1000 post-processing, pass@k, AST gate), `ds1000_runner.py` (DS-1000 tests in
`ds1000-env`), `run_arm.py` (generation + tests + upload for one arm).

In [ ]:
%%writefile evalkit.py
"""Shared helpers for the testing notebooks (generation side, standard library only).

The notebooks write this file next to themselves and import it, so every model is
prompted, post-processed and scored the same way.
"""
from __future__ import annotations

import ast
import random
import re
import textwrap
from collections import defaultdict
from math import comb

FEWSHOT_K = 3
FEWSHOT_SEED = 42
REFINE_INSTRUCTION = (
    "The code above failed when it was run against the tests. Error:\n{error}\n\n"
    "Fix the code. Reply with the corrected code only."
)


def postprocess_ds1000(code: str) -> str:
    """Official DS-1000 post-processing (test_ds1000.py, commit b39aab7)."""
    code = code.split("</code>")[0]
    code = code.replace("```python", "")
    code = code.split("```")[0]
    code = code.split("\nEND SOLUTION")[0]
    code = code.replace("<code>", "")
    return code


def ds1000_program(problem: dict, code: str) -> str:
    """Test program exactly as assembled by the official DS-1000 evaluator."""
    return (
        problem["code_context"] + "\n"
        + f"code = {repr(code)}\n"
        + "test_execution(code)\n"
        + ("test_string(code)\n" if "test_string(" in problem["code_context"] else "\n")
    )


def pass_at_k(n: int, c: int, k: int) -> float:
    """Unbiased pass@k estimator (Chen et al., 2021)."""
    if c == 0:
        return 0.0
    if n - c < k:
        return 1.0
    return 1.0 - comb(n - c, k) / comb(n, k)


def summarize_ds1000(results: list[dict], library_of: dict[int, str], k_values=(1, 10)) -> dict:
    """Per-problem and aggregate pass@k from per-sample results {problem_id, sample, passed}."""
    counts: dict[int, list[int]] = defaultdict(lambda: [0, 0])
    for row in results:
        counts[row["problem_id"]][0] += 1
        counts[row["problem_id"]][1] += int(bool(row["passed"]))
    per_problem = {
        pid: {"n": n, "c": c, **{f"pass@{k}": pass_at_k(n, c, k) for k in k_values}}
        for pid, (n, c) in sorted(counts.items())
    }
    by_library: dict[str, list[dict]] = defaultdict(list)
    for pid, scores in per_problem.items():
        by_library[library_of[pid]].append(scores)

    def mean(rows, key):
        return sum(r[key] for r in rows) / len(rows) if rows else 0.0

    rows = list(per_problem.values())
    return {
        "problems": len(rows),
        **{f"pass@{k}": mean(rows, f"pass@{k}") for k in k_values},
        "by_library": {lib: {"problems": len(r), **{f"pass@{k}": mean(r, f"pass@{k}") for k in k_values}}
                       for lib, r in sorted(by_library.items())},
        "per_problem": per_problem,
    }


def select_fewshot(rows: list[dict], k: int = FEWSHOT_K, seed: int = FEWSHOT_SEED) -> list[dict]:
    """First k of the same FEWSHOT_K examples, so a 1-shot prompt uses the first example of the 3-shot prompt."""
    return random.Random(seed).sample(rows, FEWSHOT_K)[:k]


def fewshot_ds1000_ids(n_problems: int = 1000, k: int = FEWSHOT_K, seed: int = FEWSHOT_SEED) -> list[int]:
    """DS-1000 problems used as few-shot examples; excluded from every arm's analysis."""
    return sorted(random.Random(seed).sample(range(n_problems), k))


def code4ml_messages(instruction: str, shots: list[dict] | None = None) -> list[dict]:
    """Same chat format as training: user = instruction, assistant = code."""
    messages = []
    for shot in shots or []:
        messages += [{"role": "user", "content": shot["instruction"]},
                     {"role": "assistant", "content": shot["code"]}]
    return messages + [{"role": "user", "content": instruction}]


def ds1000_messages(prompt: str, shots: list[dict] | None = None) -> list[dict]:
    messages = []
    for shot in shots or []:
        messages += [{"role": "user", "content": shot["prompt"]},
                     {"role": "assistant", "content": shot["reference_code"]}]
    return messages + [{"role": "user", "content": prompt}]


def describe_error(result: str) -> str:
    """Readable feedback from a DS-1000 execution result ("failed: <msg>" or "timed out")."""
    if result == "timed out":
        return "The code timed out."
    message = result.removeprefix("failed:").strip()
    return message or "The code ran but produced a wrong result (a test assertion failed)."


def structural_feedback(error: str, issues: list[dict]) -> str:
    """Execution error plus the SonarQube issues of the same code (refinement with structural feedback)."""
    if not issues:
        return f"{error}\n\nSonarQube found no structural issues in this code."
    lines = "\n".join(f"- line {i['line']}: {i['message']} ({i['rule']})" if i.get("line")
                      else f"- {i['message']} ({i['rule']})" for i in issues)
    return f"{error}\n\nSonarQube issues in this code:\n{lines}"


def refine_messages(history: list[dict], previous_code: str, error: str) -> list[dict]:
    return history + [{"role": "assistant", "content": previous_code},
                      {"role": "user", "content": REFINE_INSTRUCTION.format(error=error)}]


# Opening fence with any language tag; a block left open (answer cut at max_new_tokens) runs to the end.
_BLOCK = re.compile(r"```([\w+-]*)[ \t]*\n(.*?)(?:```|\Z)", re.DOTALL)
_PYTHON_TAGS = ("python", "py", "python3", "")


def python_block(completion: str) -> str | None:
    """First fenced block tagged python (or untagged); None if there is none. Skips ```sh / ```bash install lines."""
    for tag, body in _BLOCK.findall(completion):
        if tag.lower() in _PYTHON_TAGS:
            return body
    return None


def _parses_as_body(code: str) -> bool:
    """Non-empty and valid as a function body (DS-1000 answers may be an indented body that uses return)."""
    body = textwrap.indent(textwrap.dedent(code), "    ")
    return bool(code.strip()) and ast_valid(f"def _():\n{body}\n    pass")


def extract_code(completion: str) -> str:
    """Code4ML track. A bare code reply (maybe followed by a stray ``` and prose) is kept up to the first fence;
    otherwise the first python block; otherwise the whole reply (then fails the AST gate)."""
    before = completion.split("```")[0].strip()
    if before and ast_valid(before):
        return before
    block = python_block(completion)
    return (completion if block is None else block).strip()


def ds1000_code(completion: str) -> str:
    """DS-1000 track. The official post-processing first; it assumes a bare completion, so when instruct-model prose
    makes its result invalid, post-process the first python block instead."""
    official = postprocess_ds1000(completion)
    if _parses_as_body(official):
        return official
    block = python_block(completion)
    return official if block is None else postprocess_ds1000(block)


def ast_valid(code: str) -> bool:
    """Validity gate from proposal Bab III: only parseable code is rated."""
    try:
        ast.parse(code)
        return True
    except (SyntaxError, ValueError):
        return False

In [ ]:
%%writefile ds1000_runner.py
"""Run DS-1000 tests for generated answers. Executed with the DS-1000 Python 3.10 environment.

    python ds1000_runner.py PROBLEMS.jsonl ANSWERS.jsonl RESULTS.jsonl [--workers N] [--timeout S]

PROBLEMS.jsonl: {"problem_id", "code_context"} per line
ANSWERS.jsonl : {"problem_id", "sample", "code"} per line (code already post-processed)
RESULTS.jsonl : {"problem_id", "sample", "passed", "result"} per line

Uses the unmodified official execution.py (xlang-ai/DS-1000, commit b39aab7) placed next to this file.
"""
import argparse
import concurrent.futures as cfuts
import json
import multiprocessing
import os

# Same settings as the official test_ds1000.py: quiet TensorFlow, tests run on CPU only.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
# Headless matplotlib. A Jupyter kernel exports MPLBACKEND=module://matplotlib_inline..., which the test processes
# would inherit; ds1000-env has no matplotlib_inline, so every matplotlib test failed with "No module named".
os.environ["MPLBACKEND"] = "Agg"
# One thread per test process. By default TensorFlow and the BLAS libraries start a thread per host core in every
# test process; with many parallel tests that exceeds the container's thread limit and TensorFlow aborts
# ("Thread tf_Compute creation via pthread_create() failed"), which would count as a failed test.
for _var in ("TF_NUM_INTRAOP_THREADS", "TF_NUM_INTEROP_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS",
             "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_var] = "1"

import execution  # noqa: E402


def allocated_cpus():
    """CPUs this container may use (cgroup quota), not the host's core count."""
    try:
        quota, period = open("/sys/fs/cgroup/cpu.max").read().split()[:2]          # cgroup v2
        if quota != "max":
            return max(1, int(int(quota) / int(period)))
    except (OSError, ValueError):
        pass
    try:
        quota = int(open("/sys/fs/cgroup/cpu/cpu.cfs_quota_us").read())            # cgroup v1
        period = int(open("/sys/fs/cgroup/cpu/cpu.cfs_period_us").read())
        if quota > 0:
            return max(1, quota // period)
    except (OSError, ValueError):
        pass
    if hasattr(os, "sched_getaffinity"):
        return len(os.sched_getaffinity(0))
    return os.cpu_count() or 1


def build_program(code_context, code):
    return (
        code_context + "\n"
        + f"code = {repr(code)}\n"
        + "test_execution(code)\n"
        + ("test_string(code)\n" if "test_string(" in code_context else "\n")
    )


def run(problems_path, answers_path, results_path, workers, timeout):
    with open(problems_path) as handle:
        context = {row["problem_id"]: row["code_context"] for row in map(json.loads, handle)}
    with open(answers_path) as handle:
        answers = [json.loads(line) for line in handle]

    results = []
    with cfuts.ProcessPoolExecutor(max_workers=workers) as pool:
        futures = {
            pool.submit(execution.check_correctness,
                        build_program(context[a["problem_id"]], a["code"]), timeout, i): a
            for i, a in enumerate(answers)
        }
        for done, future in enumerate(cfuts.as_completed(futures), 1):
            answer, outcome = futures[future], future.result()
            results.append({"problem_id": answer["problem_id"], "sample": answer["sample"],
                            "passed": outcome["passed"], "result": outcome["result"]})
            if done % 500 == 0 or done == len(futures):
                print(f"tested {done}/{len(futures)}", flush=True)

    results.sort(key=lambda r: (r["problem_id"], r["sample"]))
    with open(results_path, "w") as handle:
        for row in results:
            handle.write(json.dumps(row) + "\n")


if __name__ == "__main__":
    # execution.py runs each test in a nested local function, which needs the fork start method.
    multiprocessing.set_start_method("fork", force=True)
    parser = argparse.ArgumentParser()
    parser.add_argument("problems")
    parser.add_argument("answers")
    parser.add_argument("results")
    parser.add_argument("--workers", type=int, default=min(16, allocated_cpus()))
    parser.add_argument("--timeout", type=float, default=120.0)
    args = parser.parse_args()
    print(f"testing with {args.workers} parallel workers", flush=True)
    run(args.problems, args.answers, args.results, args.workers, args.timeout)

In [ ]:
%%writefile run_arm.py
"""Evaluate one arm, or one refinement run, and push the results to the Hub.

Called by the testing entry points, one process per arm so GPU memory is fully released between models (index.py and
index.ipynb run it inside src/ with ../config.yml; the standalone notebooks with config.yml next to them):

    python run_arm.py config.yml generate lora
    python run_arm.py config.yml refine lora 43
    python run_arm.py config.yml refine-sonar lora 43   # + SonarQube issues in the feedback (needs SonarQube)
    python run_arm.py config.yml retest vanilla    # re-extract code + rerun DS-1000 tests, no generation

Exit code 0 = all runs of this arm are on the Hub; 1 = failed (traceback uploaded to OUTPUT_REPO/errors/).
"""
import json
import os
import random
import subprocess
import sys
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path

import yaml

import evalkit

EXTRACTION = "python_block"   # how code is taken from a reply (evalkit.python_block); retest skips runs that match
BASE_7B = "Qwen/Qwen2.5-Coder-7B-Instruct"
BASE_14B = "Qwen/Qwen2.5-Coder-14B-Instruct"
ARMS = {   # shots = training examples in the prompt (evalkit.select_fewshot / fewshot_ds1000_ids)
    "vanilla": dict(base=BASE_7B, adapter=None, seeds=[None], quant=False, shots=0),
    "few-shot": dict(base=BASE_7B, adapter=None, seeds=[None], quant=False, shots=3),
    "lora": dict(base=BASE_7B, adapter="dimsmauls/qwen2.5-coder-7b-code4ml-lora", seeds=[42, 43, 44],
                 quant=False, shots=0),
    # Trained on a 4-bit NF4 base; evaluated on the bf16 base like LoRA, since vLLM >= 0.30 dropped bitsandbytes and
    # every arm must run on the same engine.
    "qlora": dict(base=BASE_7B, adapter="dimsmauls/qwen2.5-coder-7b-code4ml-qlora", seeds=[42, 43, 44],
                  quant=False, shots=0),
    "qwen-14b": dict(base=BASE_14B, adapter=None, seeds=[None], quant=False, shots=0),
}
# LoRA hyperparameter sensitivity (training/hyperparameter/index.ipynb): one changed hyperparameter per adapter, seed 42, stored
# in HPARAM_REPO under lora-<config>/seed-42/. Evaluated as arm hp-lora-<config>.
HPARAM_REPO = "dimsmauls/qwen2.5-coder-7b-code4ml-hparam"
for _config, _rank in (("cosine", 8), ("r32", 32), ("lr1e-4", 8)):
    ARMS[f"hp-lora-{_config}"] = dict(base=BASE_7B, adapter=HPARAM_REPO, adapter_dir=f"lora-{_config}",
                                      seeds=[42], quant=False, shots=0, max_lora_rank=max(16, _rank))
# Prompting shots: every model again with 1 and 3 examples in the prompt, as arm <model>-<k>shot. 0 shots are the arms
# above; vanilla with 3 shots is the existing few-shot arm.
for _model in ("vanilla", "qwen-14b", "lora", "qlora", "hp-lora-cosine", "hp-lora-r32", "hp-lora-lr1e-4"):
    for _shots in (1, 3):
        if (_model, _shots) != ("vanilla", 3):
            ARMS[f"{_model}-{_shots}shot"] = {**ARMS[_model], "shots": _shots}


def prefix(cfg):
    """Pilot runs go to pilot/ so they never block or overwrite the full evaluation."""
    return "pilot/" if cfg.get("pilot_problems") else ""


def run_name(cfg, kind, seed):
    return prefix(cfg) + (kind if seed is None else f"{kind}/seed-{seed}")


def write_jsonl(path, rows):
    with open(path, "w") as handle:
        for row in rows:
            handle.write(json.dumps(row) + "\n")


def load_data(cfg):
    from datasets import load_dataset

    code4ml = load_dataset(cfg["dataset"], "training", split="train")
    data = {
        "code4ml_test": [row for row in code4ml if row["split"] == "test"],
        "code4ml_train": [row for row in code4ml if row["split"] == "train"],
    }
    problems = {int(p["metadata"]["problem_id"]): p for p in load_dataset("xlangai/DS-1000", split="test")}
    fewshot_ids = evalkit.fewshot_ds1000_ids(len(problems))   # examples; excluded from every arm
    eval_ids = [pid for pid in sorted(problems) if pid not in fewshot_ids]
    if cfg.get("pilot_problems"):
        data["code4ml_test"] = data["code4ml_test"][:cfg["pilot_problems"]]
        eval_ids = eval_ids[:cfg["pilot_problems"]]
    write_jsonl("ds1000_problems.jsonl",
                [{"problem_id": pid, "code_context": p["code_context"]} for pid, p in problems.items()])
    data.update(problems=problems, fewshot_ids=fewshot_ids, eval_ids=eval_ids,
                library_of={pid: p["metadata"]["library"] for pid, p in problems.items()})
    return data


class Engine:
    """One vLLM engine per process; LoRA adapters are attached per request."""

    def __init__(self, arm, gen):
        from vllm import LLM

        import torch
        import vllm

        self.arm = arm
        self.llm = LLM(model=arm["base"], dtype="bfloat16", max_model_len=gen["max_model_len"],
                       gpu_memory_utilization=0.90, seed=gen["seed"], enable_lora=arm["adapter"] is not None,
                       max_lora_rank=arm.get("max_lora_rank", 16),
                       **({"quantization": "bitsandbytes"} if arm["quant"] else {}))
        self.tokenizer = self.llm.get_tokenizer()
        self.gpu = torch.cuda.get_device_name(0)
        self.versions = {"vllm": vllm.__version__, "torch": torch.__version__}

    def prompt(self, messages):
        return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def lora(self, seed):
        if self.arm["adapter"] is None:
            return None
        from huggingface_hub import snapshot_download
        from vllm.lora.request import LoRARequest

        folder = f"{self.arm['adapter_dir']}/seed-{seed}" if self.arm.get("adapter_dir") else f"seed-{seed}"
        local = snapshot_download(self.arm["adapter"], allow_patterns=[f"{folder}/*"])
        return LoRARequest(folder, seed, f"{local}/{folder}")

    def generate(self, prompts, request, **sampling):
        from vllm import SamplingParams

        return self.llm.generate(prompts, SamplingParams(**sampling), lora_request=request)


def run_ds1000_tests(answers, tag):
    answers_path, results_path = f"answers_{tag}.jsonl", f"results_{tag}.jsonl"
    write_jsonl(answers_path, [{k: row[k] for k in ("problem_id", "sample", "code")} for row in answers])
    subprocess.run(["ds1000-env/bin/python", "ds1000_runner.py", "ds1000_problems.jsonl",
                    answers_path, results_path], check=True)
    with open(results_path) as handle:
        return [json.loads(line) for line in handle]


def evaluate(engine, data, cfg, api, kind, seed):
    arm, gen = ARMS[kind], cfg["generation"]
    name = run_name(cfg, kind, seed)
    out = Path("results") / name
    out.mkdir(parents=True, exist_ok=True)
    request = engine.lora(seed)
    started = time.time()

    # Structural track: one greedy answer per Code4ML test instruction.
    shots = evalkit.select_fewshot(data["code4ml_train"], arm["shots"]) if arm["shots"] else []
    prompts = [engine.prompt(evalkit.code4ml_messages(row["instruction"], shots)) for row in data["code4ml_test"]]
    outputs = engine.generate(prompts, request, temperature=0.0, max_tokens=gen["max_new_tokens"])
    code4ml_rows = []
    for row, output in zip(data["code4ml_test"], outputs):
        completion = output.outputs[0].text
        code = evalkit.extract_code(completion)
        code4ml_rows.append({"id": row["id"], "graph_vertex_class": row["graph_vertex_class"],
                             "graph_vertex_subclass": row["graph_vertex_subclass"],
                             "instruction": row["instruction"], "completion": completion,
                             "code": code, "ast_valid": evalkit.ast_valid(code)})
    write_jsonl(out / "code4ml_outputs.jsonl", code4ml_rows)

    # Functional track: sampled answers per DS-1000 problem.
    problems, eval_ids = data["problems"], data["eval_ids"]
    ds_shots = [problems[pid] for pid in data["fewshot_ids"][:arm["shots"]]]   # first k of the 3 excluded problems
    prompts = [engine.prompt(evalkit.ds1000_messages(problems[pid]["prompt"], ds_shots)) for pid in eval_ids]
    outputs = engine.generate(prompts, request, n=gen["ds1000_samples"], temperature=gen["ds1000_temperature"],
                              top_p=gen["ds1000_top_p"], max_tokens=gen["max_new_tokens"], seed=gen["seed"])
    generations = [{"problem_id": pid, "sample": i, "completion": o.text, "code": evalkit.ds1000_code(o.text)}
                   for pid, output in zip(eval_ids, outputs) for i, o in enumerate(output.outputs)]
    write_jsonl(out / "ds1000_generations.jsonl", generations)
    generation_minutes = round((time.time() - started) / 60, 1)

    results = run_ds1000_tests(generations, name.replace("/", "_"))
    write_jsonl(out / "ds1000_results.jsonl", results)
    ds_summary = evalkit.summarize_ds1000(results, data["library_of"])
    with open(out / "ds1000_per_problem.json", "w") as handle:
        json.dump(ds_summary.pop("per_problem"), handle)

    valid = sum(r["ast_valid"] for r in code4ml_rows)
    summary = {
        "run": name, "base_model": arm["base"], "adapter_repo": arm["adapter"], "seed": seed,
        "quantized_4bit": arm["quant"], "fewshot": arm["shots"] > 0, "shots": arm["shots"],
        "fewshot_ds1000_ids": data["fewshot_ids"],
        "gpu": engine.gpu, "versions": engine.versions, "pilot_problems": cfg.get("pilot_problems"),
        "generation": {**gen, "code4ml": "greedy"}, "code_extraction": EXTRACTION,
        "code4ml": {"answers": len(code4ml_rows), "ast_valid": valid, "ast_valid_rate": valid / len(code4ml_rows)},
        "ds1000": ds_summary,
        "generation_minutes": generation_minutes,
        "total_minutes": round((time.time() - started) / 60, 1),
    }
    with open(out / "summary.json", "w") as handle:
        json.dump(summary, handle, indent=2)
    api.upload_folder(repo_id=cfg["output_repo"], repo_type="dataset", folder_path=str(out),
                      path_in_repo=name, commit_message=f"evaluation {name}")
    print(f"{name}: AST valid {valid}/{len(code4ml_rows)} | DS-1000 pass@1 {ds_summary['pass@1']:.4f} "
          f"pass@10 {ds_summary['pass@10']:.4f} | {summary['total_minutes']} min", flush=True)


def retest(data, cfg, api, kind, seed):
    """Re-extract the code from the uploaded completions and rerun the DS-1000 tests (no generation, no GPU).

    For runs generated before evalkit.python_block: replaces the code / ast_valid fields of code4ml_outputs.jsonl and
    ds1000_generations.jsonl (completions stay as they are), the DS-1000 results and summary.json. The previous scores
    are kept in summary.json under "retested". Runs that already use python_block are skipped."""
    from huggingface_hub import hf_hub_download

    name = run_name(cfg, kind, seed)
    fetch = lambda file: hf_hub_download(cfg["output_repo"], f"{name}/{file}", repo_type="dataset")
    with open(fetch("summary.json")) as handle:
        summary = json.load(handle)
    if summary.get("code_extraction") == EXTRACTION:
        print(f"{name}: already uses {EXTRACTION}, skipping", flush=True)
        return
    with open(fetch("code4ml_outputs.jsonl")) as handle:
        code4ml_rows = [json.loads(line) for line in handle]
    with open(fetch("ds1000_generations.jsonl")) as handle:
        generations = [json.loads(line) for line in handle]
    out = Path("results") / name
    out.mkdir(parents=True, exist_ok=True)
    started = time.time()

    for row in code4ml_rows:
        row["code"] = evalkit.extract_code(row["completion"])
        row["ast_valid"] = evalkit.ast_valid(row["code"])
    write_jsonl(out / "code4ml_outputs.jsonl", code4ml_rows)
    for row in generations:
        row["code"] = evalkit.ds1000_code(row["completion"])
    write_jsonl(out / "ds1000_generations.jsonl", generations)

    results = run_ds1000_tests(generations, name.replace("/", "_"))
    write_jsonl(out / "ds1000_results.jsonl", results)
    ds_summary = evalkit.summarize_ds1000(results, data["library_of"])
    with open(out / "ds1000_per_problem.json", "w") as handle:
        json.dump(ds_summary.pop("per_problem"), handle)

    valid = sum(r["ast_valid"] for r in code4ml_rows)
    previous = {"ds1000_pass@1": summary["ds1000"]["pass@1"], "ds1000_pass@10": summary["ds1000"]["pass@10"],
                "code4ml_ast_valid": summary["code4ml"]["ast_valid"]}
    summary["code4ml"] = {"answers": len(code4ml_rows), "ast_valid": valid, "ast_valid_rate": valid / len(code4ml_rows)}
    summary["ds1000"] = ds_summary
    summary["code_extraction"] = EXTRACTION
    summary["retested"] = {"at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                           "minutes": round((time.time() - started) / 60, 1), "previous": previous}
    with open(out / "summary.json", "w") as handle:
        json.dump(summary, handle, indent=2)
    api.upload_folder(repo_id=cfg["output_repo"], repo_type="dataset", folder_path=str(out),
                      path_in_repo=name, commit_message=f"retest {name}")
    print(f"{name}: AST valid {previous['code4ml_ast_valid']} -> {valid}/{len(code4ml_rows)} | DS-1000 pass@1 "
          f"{previous['ds1000_pass@1']:.4f} -> {ds_summary['pass@1']:.4f} | pass@10 "
          f"{previous['ds1000_pass@10']:.4f} -> {ds_summary['pass@10']:.4f}", flush=True)


def refine(engine, data, cfg, api, method, seed, structural=False):
    """Refinement with execution feedback. structural=True adds the SonarQube issues of the failing code to the
    feedback (stop rule unchanged: a problem leaves the loop once its tests pass) and records SonarQube measures and
    issues of every answer."""
    ref, gen = cfg["refinement-loop"], cfg["generation"]
    name = run_name(cfg, f"{'refinement-sonar' if structural else 'refinement'}-{method}", seed)
    if structural:
        import sonar
    out = Path("results") / name
    out.mkdir(parents=True, exist_ok=True)
    request = engine.lora(seed)
    subset = sorted(random.Random(42).sample(data["eval_ids"], min(ref["problems"], len(data["eval_ids"]))))
    started = time.time()

    histories = {pid: evalkit.ds1000_messages(data["problems"][pid]["prompt"]) for pid in subset}
    open_ids, rows = list(subset), []
    for iteration in range(ref["max_iterations"] + 1):
        outputs = engine.generate([engine.prompt(histories[pid]) for pid in open_ids], request,
                                  temperature=0.0, max_tokens=gen["max_new_tokens"])
        answers = [{"problem_id": pid, "sample": 0, "completion": o.outputs[0].text,
                    "code": evalkit.ds1000_code(o.outputs[0].text)} for pid, o in zip(open_ids, outputs)]
        results = {r["problem_id"]: r for r in run_ds1000_tests(answers, f"refine_{method}_it{iteration}")}
        scans = sonar.scan({a["problem_id"]: a["code"] for a in answers}) if structural else {}
        still_open = []
        for answer in answers:
            pid, result = answer["problem_id"], results[answer["problem_id"]]
            rows.append({"problem_id": pid, "iteration": iteration, "passed": result["passed"],
                         "result": result["result"], "completion": answer["completion"], "code": answer["code"],
                         **({"sonar": scans[pid]} if structural else {})})
            if not result["passed"]:
                feedback = evalkit.describe_error(result["result"])
                if structural:
                    feedback = evalkit.structural_feedback(feedback, scans[pid]["issues"])
                histories[pid] = evalkit.refine_messages(histories[pid], answer["completion"], feedback)
                still_open.append(pid)
        print(f"iteration {iteration}: solved {len(subset) - len(still_open)}/{len(subset)}", flush=True)
        open_ids = still_open
        if not open_ids:
            break

    write_jsonl(out / "refinement_results.jsonl", rows)
    first_pass = {}
    for row in rows:
        if row["passed"]:
            first_pass.setdefault(row["problem_id"], row["iteration"])
    cumulative = [sum(1 for it in first_pass.values() if it <= i) / len(subset)
                  for i in range(ref["max_iterations"] + 1)]
    summary = {
        "run": name, "method": method, "seed": seed, "adapter_repo": ARMS[method]["adapter"],
        "quantized_4bit": ARMS[method]["quant"], "problems": len(subset), "problem_ids": subset,
        "max_iterations": ref["max_iterations"], "pass_rate_by_iteration": cumulative, "decoding": "greedy",
        "code_extraction": EXTRACTION, "structural_feedback": structural,
        "gpu": engine.gpu, "versions": {**engine.versions, **({"sonarqube": sonar.version()} if structural else {})},
        "minutes": round((time.time() - started) / 60, 1),
    }
    with open(out / "summary.json", "w") as handle:
        json.dump(summary, handle, indent=2)
    api.upload_folder(repo_id=cfg["output_repo"], repo_type="dataset", folder_path=str(out),
                      path_in_repo=name, commit_message=f"refinement {name}")
    print(f"{name}: pass rate by iteration {[round(x, 4) for x in cumulative]}", flush=True)


def main(argv):
    config_path, mode, kind = argv[:3]
    cfg = yaml.safe_load(open(config_path))
    from huggingface_hub import HfApi

    api = HfApi()
    label = {"generate": kind, "refine": f"refinement-{kind}", "refine-sonar": f"refinement-sonar-{kind}",
             "retest": f"retest-{kind}"}[mode]
    try:
        if mode == "retest":
            data = load_data(cfg)
            for seed in ARMS[kind]["seeds"]:
                retest(data, cfg, api, kind, seed)
            return 0
        on_hub = set(api.list_repo_files(cfg["output_repo"], repo_type="dataset"))
        if mode == "generate":
            todo = [s for s in ARMS[kind]["seeds"] if f"{run_name(cfg, kind, s)}/summary.json" not in on_hub]
            if not todo:
                print(f"{kind}: already on the Hub, skipping", flush=True)
                return 0
        else:
            seed = int(argv[3])
            if f"{run_name(cfg, label, seed)}/summary.json" in on_hub:
                print(f"{run_name(cfg, label, seed)}: already on the Hub, skipping", flush=True)
                return 0
        data = load_data(cfg)
        engine = Engine(ARMS[kind], cfg["generation"])
        if mode == "generate":
            for seed in todo:
                evaluate(engine, data, cfg, api, kind, seed)
        else:
            refine(engine, data, cfg, api, kind, seed, structural=mode == "refine-sonar")
        return 0
    except Exception:
        error = traceback.format_exc()
        print(error, flush=True)
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        try:
            api.upload_file(path_or_fileobj=error.encode(), path_in_repo=f"errors/{label}-{stamp}.txt",
                            repo_id=cfg["output_repo"], repo_type="dataset", commit_message=f"{label} error {stamp}")
        except Exception as upload_exc:  # the exit code still reports the failure
            print("could not upload the error log:", upload_exc, flush=True)
        return 1


if __name__ == "__main__":
    sys.exit(main(sys.argv[1:]))

## 3. Read `config.yml`, check the GPU, log in

In [ ]:
import json, os, subprocess, sys
import yaml
from huggingface_hub import HfApi, hf_hub_download, login

cfg = yaml.safe_load(open("config.yml"))
# Credentials from config.yml fill in only what the environment does not already provide; child processes inherit them.
for key, env in (("hf_token", "HF_TOKEN"), ("vast_api_key", "VAST_API_KEY")):
    if cfg.get(key) and not os.environ.get(env):
        os.environ[env] = str(cfg[key])
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")), "| VAST_API_KEY set:", bool(os.environ.get("VAST_API_KEY")))
arms = [kind for kind, enabled in cfg["arms"].items() if enabled]
refine_methods = [m for m in ("lora", "qlora") if cfg["refinement-loop"].get(m)]
retest_arms = [kind for kind, enabled in cfg.get("retest", {}).items() if enabled]
PREFIX = "pilot/" if cfg.get("pilot_problems") else ""
print("arms:", arms, "| refinement:", refine_methods, "| retest:", retest_arms, "| pilot:", cfg.get("pilot_problems"))

if arms or refine_methods:  # retest alone runs on CPU
    # nvidia-smi instead of torch, so this process does not hold GPU memory the arms need
    name, total_mb = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                                    capture_output=True, text=True, check=True).stdout.strip().split(", ")
    print(f"GPU: {name}, {int(total_mb) / 1024:.0f} GB")
    if any(kind.startswith("qwen-14b") for kind in arms) and int(total_mb) < 40 * 1024:
        raise SystemExit("qwen-14b needs a GPU with >= 40 GB: disable it in config.yml or rent a bigger GPU")

token = os.environ.get("HF_TOKEN")
login(token=token) if token else login()
api = HfApi()
api.create_repo(cfg["output_repo"], repo_type="dataset", private=True, exist_ok=True)

def stream(cmd):
    """Run a command and show its output live in the notebook."""
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    return process.wait()

failed = []

## 4. Run the arms
Each arm runs in its own process (fresh GPU memory) and uploads every finished run before the next starts. A failed
arm uploads its traceback to `errors/` and the next arm still runs.

In [ ]:
for kind in arms:
    print(f"\n########## {kind} ##########", flush=True)
    if stream([sys.executable, "run_arm.py", "config.yml", "generate", kind]) != 0:
        failed.append(kind)
print("failed arms:", failed)

## 4b. Retest (optional)
Arms enabled under `retest:` take the code again from the replies already on the Hub (first python block, see
`evalkit.python_block`), recompute AST validity and rerun the DS-1000 tests; no generation, no GPU. Old scores are kept
in `summary.json` under `retested`. Runs generated with the current extraction are skipped.

In [ ]:
for kind in retest_arms:
    print(f"\n########## retest {kind} ##########", flush=True)
    if stream([sys.executable, "run_arm.py", "config.yml", "retest", kind]) != 0:
        failed.append(f"retest-{kind}")

## 5. Iterative refinement
`seed: auto` picks the median seed of that method by DS-1000 pass@1 (from its `summary.json` files on the Hub).

In [ ]:
def median_seed(method):
    scores = []
    for seed in (42, 43, 44):
        path = hf_hub_download(cfg["output_repo"], f"{PREFIX}{method}/seed-{seed}/summary.json", repo_type="dataset")
        scores.append((json.load(open(path))["ds1000"]["pass@1"], seed))
    scores.sort()
    print(method, "pass@1 by seed:", {seed: round(score, 4) for score, seed in scores})
    return scores[1][1]

for method in refine_methods:
    print(f"\n########## refinement-{method} ##########", flush=True)
    try:
        seed_setting = cfg["refinement-loop"]["seed"]
        seed = median_seed(method) if seed_setting == "auto" else int(seed_setting)
    except Exception as exc:
        print(f"cannot pick the seed for {method} (are its results on the Hub?):", exc)
        failed.append(f"refinement-{method}")
        continue
    print("refinement seed:", seed)
    if stream([sys.executable, "run_arm.py", "config.yml", "refine", method, str(seed)]) != 0:
        failed.append(f"refinement-{method}")
print("failed:", failed)

## 6. Shut the machine down
Runs whether everything succeeded or something failed (failures are in `errors/` on the Hub). Colab: releases the
runtime. Vast.ai: `vastai stop instance` (GPU billing stops; only the small disk fee remains) — needs `VAST_API_KEY`.
If you stop a cell yourself, this cell does not run: stop the machine yourself.

In [ ]:
def shutdown():
    try:
        from google.colab import runtime  # only importable on Colab
        print("releasing the Colab runtime")
        runtime.unassign()
        return
    except ImportError:
        pass
    container_id, api_key = os.environ.get("CONTAINER_ID"), os.environ.get("VAST_API_KEY")
    if not (container_id and api_key):
        print("CONTAINER_ID or VAST_API_KEY not set: STOP THE INSTANCE YOURSELF on cloud.vast.ai")
        return
    subprocess.run(["pip", "install", "-q", "vastai"], check=True)
    print(f"stopping Vast.ai instance {container_id}")
    subprocess.run(["vastai", "stop", "instance", container_id, "--api-key", api_key], check=True)

print("done; failed:", failed or "none")
print(f"results: https://huggingface.co/datasets/{cfg['output_repo']}")
if cfg.get("auto_shutdown", True):
    shutdown()
else:
    print("auto_shutdown is off: stop the machine yourself")